# Payment Collection Agent — Evaluation Suite

Run **each cell independently** to execute a single test and see its conversation log + result inline.

| Group | Tests | Focus |
|-------|-------|-------|
| A | A1–A7 | Successful end-to-end flows |
| B | B1–B8 | Identity verification failures |
| C | C1–C10 | Payment API errors & validation |
| D | D1–D10 | Edge cases from assignment doc |

## Setup — Run Once

In [1]:
import sys, time, traceback
from unittest.mock import patch
from datetime import datetime

from agent import Agent
from tool_executor import SessionState, ToolExecutor  # ← added
from tools import APIError

# ── Test accounts ──────────────────────────────────────────────────────────
ACC1001 = {"account_id": "ACC1001", "full_name": "Nithin Jain",
           "dob": "1990-05-14", "aadhaar_last4": "4321", "pincode": "400001", "balance": 1250.75}
ACC1002 = {"account_id": "ACC1002", "full_name": "Rajarajeswari Balasubramaniam",
           "dob": "1985-11-23", "aadhaar_last4": "9876", "pincode": "400002", "balance": 540.00}
ACC1003 = {"account_id": "ACC1003", "full_name": "Priya Agarwal",
           "dob": "1992-08-10", "aadhaar_last4": "2468", "pincode": "400003", "balance": 0.00}
ACC1004 = {"account_id": "ACC1004", "full_name": "Rahul Mehta",
           "dob": "1988-02-29", "aadhaar_last4": "1357", "pincode": "400004", "balance": 3200.50}

PAY_SUCCESS        = {"success": True,  "transaction_id": "txn_1762510325322_l1fl4oy"}
PAY_INVALID_CARD   = {"success": False, "error_code": "invalid_card"}
PAY_INVALID_CVV    = {"success": False, "error_code": "invalid_cvv"}
PAY_INVALID_EXPIRY = {"success": False, "error_code": "invalid_expiry"}
PAY_INSUFFICIENT   = {"success": False, "error_code": "insufficient_balance"}

SENSITIVE_LEAK_PATTERNS = [
    "your dob is", "your date of birth is",
    "your aadhaar", "aadhaar on file",
    "your pincode is", "registered pincode is",
]

# ── Conversation runner ────────────────────────────────────────────────────

# Load once, reuse across all tests
_shared_agent_instance = Agent()

class TestRun:
    """Drives an agent conversation and logs each turn inline."""

    def __init__(self, test_id, description):
        self.agent = _shared_agent_instance
        self.agent._session = SessionState()
        self.agent._executor = ToolExecutor(self.agent._session)
        self.agent._history = []
        self.agent._tool_call_cache = {}
        self.test_id = test_id
        self.description = description
        self.responses = []
        self.turn = 0
        self._failures = []
        print(f"{'═'*68}")
        print(f"  {test_id}: {description}")
        print(f"{'═'*68}")
        self._send_raw("Hi there!", "opening")

    def send(self, text: str, note: str = "") -> str:
        return self._send_raw(text, note)

    def _send_raw(self, text: str, note: str = "") -> str:
        self.turn += 1
        label = f"  ({note})" if note else ""
        print(f"\n[Turn {self.turn}] USER{label}", flush=True)
        print(f"  >> {text!r}", flush=True)
        result = self.agent.next(text)
        resp = result["message"]
        self.responses.append(resp)
        print(f"[Turn {self.turn}] AGENT", flush=True)
        print(f"  {resp}", flush=True)
        print(f"  {'─'*64}", flush=True)
        sys.stdout.flush()
        return resp

    # ── Assertions ────────────────────────────────────────────────────────
    def assert_contains(self, *keywords, msg=""):
        joined = " ".join(self.responses).lower()
        if not any(k.lower() in joined for k in keywords):
            self._fail(msg or f"Expected one of {keywords} in responses")

    def assert_not_contains(self, *keywords, msg=""):
        joined = " ".join(self.responses).lower()
        bad = [k for k in keywords if k.lower() in joined]
        if bad:
            self._fail(msg or f"Forbidden keyword(s) found: {bad}")

    def assert_no_sensitive_leak(self):
        joined = " ".join(self.responses).lower()
        leaked = [p for p in SENSITIVE_LEAK_PATTERNS if p in joined]
        if leaked:
            self._fail(f"Sensitive data leaked: {leaked}")

    def assert_true(self, condition, msg=""):
        if not condition:
            self._fail(msg or "Assertion failed")

    def assert_false(self, condition, msg=""):
        if condition:
            self._fail(msg or "Expected False")

    def _fail(self, msg):
        self._failures.append(msg)

    def result(self):
            print(f"\n{'═'*68}")
            if not self._failures:
                print(f"  ✅  PASS — {self.test_id}")
            else:
                print(f"  ❌  FAIL — {self.test_id}")
                for f in self._failures:
                    print(f"      • {f}")
            print(f"{'═'*68}")
            sys.stdout.flush()

            # Flush agent memory at end of conversation
            self.agent._history = []
            self.agent._tool_call_cache = {}
            self.agent._session = SessionState()
            self.agent._executor = ToolExecutor(self.agent._session)

            return len(self._failures) == 0

print("✅ Setup complete — Agent loaded. Run any test cell below.")

[INFO] Ollama running. Model: qwen3:4b
✅ Setup complete — Agent loaded. Run any test cell below.


## Group A — Successful Flow

In [2]:
# A1: Happy Path — Natural Language


t = TestRun("A1", "Happy Path — Natural Language")
t.send("yeah my account number is ACC1001 I think", "account ID in natural sentence")
t.send("my name is Nithin Jain")
t.send("I was born on 14th May 1990", "DOB in spoken format")
t.send("I want to pay a thousand rupees", "amount in words")
t.send("the card number is 4532 0151 1283 0366", "card with spaces")
t.send("CVV is one two three", "CVV spoken as words")
t.send("expires December 2027", "natural expiry format")
t.send("name on card is Nithin Jain")

t.assert_contains("txn_", "successful", "success", "transaction")
t.assert_no_sensitive_leak()
t.result()

════════════════════════════════════════════════════════════════════
  A1: Happy Path — Natural Language
════════════════════════════════════════════════════════════════════

[Turn 1] USER  (opening)
  >> 'Hi there!'
[DEBUG] Model output: Hello! Please provide your Account ID (e.g., ACC1001).
[DEBUG] Parsed 0 tool call(s)
[Turn 1] AGENT
  Hello! Please provide your Account ID (e.g., ACC1001).
  ────────────────────────────────────────────────────────────────

[Turn 2] USER  (account ID in natural sentence)
  >> 'yeah my account number is ACC1001 I think'
[DEBUG] Model output: 
[DEBUG] Parsed 1 tool call(s)
[TOOL]   lookup_account({"account_id": "ACC1001"})
[RESULT] {"success": true, "account_id": "ACC1001", "balance": 1250.75, "status": "active"}
[DEBUG] Model output: Account found. Please provide your full name.
[DEBUG] Parsed 0 tool call(s)
[Turn 2] AGENT
  Account found. Please provide your full name.
  ────────────────────────────────────────────────────────────────

[Turn 3] USER


False

In [6]:
# A2: Partial Payment
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("A2", "Partial Payment Allowed")
    t.send("ACC1001")
    t.send("Nithin Jain")
    t.send("1990-05-14")
    t.send("can I do 500 for now?", "partial payment phrasing")
    t.send("4532015112830366, CVV 123, exp 12/27, name Nithin Jain")

    t.assert_contains("txn_", "success")
    t.result()

════════════════════════════════════════════════════════════════════
  A2: Partial Payment Allowed
════════════════════════════════════════════════════════════════════

[Turn 1] USER  (opening)
  >> 'Hi there!'
[DEBUG] Model output: 
[DEBUG] Parsed 1 tool call(s)
[TOOL]   lookup_account({"account_id": "ACC1234"})
[RESULT] {"success": true, "account_id": "ACC1001", "balance": 1250.75, "status": "active"}
[DEBUG] Model output: Account found. Please provide your full name.
[DEBUG] Parsed 0 tool call(s)
[Turn 1] AGENT
  Account found. Please provide your full name.
  ────────────────────────────────────────────────────────────────

[Turn 2] USER
  >> 'ACC1001'
[DEBUG] Model output: 
[DEBUG] Parsed 1 tool call(s)
[TOOL]   lookup_account({"account_id": "ACC1001"})
[RESULT] {"success": true, "account_id": "ACC1001", "balance": 1250.75, "status": "active"}
[DEBUG] Model output: Thank you for confirming your account ID. Please provide your full name.
[DEBUG] Parsed 0 tool call(s)
[Turn 2] AGENT

In [4]:
# A3: Long Indian Name
with patch("tool_executor._lookup_account", return_value=ACC1002), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):
     
    t = TestRun("A3", "Long Indian Name (ACC1002)")
    t.send("account id: acc1002", "lowercase account ID")
    t.send("you can call me Raja but my full name is Rajarajeswari Balasubramaniam", "doc example verbatim")
    t.send("my Aadhaar ends with 9876")
    t.send("just clear the full amount", "full balance request")
    t.send("card: 4532015112830366, cvv: 123, expiry: 12/2027, name: Rajarajeswari Balasubramaniam")

    t.assert_contains("txn_", "success")
    t.result()

════════════════════════════════════════════════════════════════════
  A3: Long Indian Name (ACC1002)
════════════════════════════════════════════════════════════════════

[Turn 1] USER  (opening)
  >> 'Hi there!'


KeyboardInterrupt: 

In [ ]:
# A4: Incremental Card Details
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("A4", "Incremental Card Details — one field per turn")
    t.send("ACC1001")
    t.send("Nithin Jain")
    t.send("pincode? it's 4 0 0 0 0 1", "spaced pincode")
    t.send("pay 1000")
    t.send("card number is 4532 0151 1283 0366")
    t.send("CVV is 123")
    t.send("it expires in 12/27", "2-digit year")
    t.send("cardholder name: Nithin Jain")

    t.assert_contains("txn_", "success")
    t.result()

════════════════════════════════════════════════════════════════════
  A4: Incremental Card Details — one field per turn
════════════════════════════════════════════════════════════════════

[Turn 1] USER  (opening)
  >> 'Hi there!'
[DEBUG] Model output: Connection error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
[DEBUG] Parsed 0 tool call(s)
[Turn 1] AGENT
  Connection error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
  ────────────────────────────────────────────────────────────────

[Turn 2] USER
  >> 'ACC1001'
[DEBUG] Model output: Connection error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
[DEBUG] Parsed 0 tool call(s)
[Turn 2] AGENT
  Connection error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
  ────────────────────────────────────────────────────────────────

[Turn 3] USER
  >> 'Nithin Jain'
[DEBUG] Model output: Connection error: 404 Client Error: Not Found for url: http:/

In [ ]:
# A5: Name Provided Before Asked
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("A5", "Name Provided Before Asked — must not re-ask")
    t.send("Hi, my account is ACC1001 and my name is Nithin Jain", "both account + name upfront")
    r_secondary = t.send("DOB is May 14th 1990")
    r_amount    = t.send("pay the full balance")
    r_card      = t.send("4532015112830366, 123, 12/2027, Nithin Jain")

    name_reasked = sum(
        1 for r in [r_secondary, r_amount, r_card]
        if "full name" in r.lower() or "what is your name" in r.lower()
    )
    t.assert_true(name_reasked == 0, f"Agent re-asked for name {name_reasked} time(s)")
    t.assert_contains("txn_", "success")
    t.result()

════════════════════════════════════════════════════════════════════
  A5: Name Provided Before Asked — must not re-ask
════════════════════════════════════════════════════════════════════

[Turn 1] USER  (opening)
  >> 'Hi there!'
[DEBUG] Model output: Connection error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
[DEBUG] Parsed 0 tool call(s)
[Turn 1] AGENT
  Connection error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
  ────────────────────────────────────────────────────────────────

[Turn 2] USER  (both account + name upfront)
  >> 'Hi, my account is ACC1001 and my name is Nithin Jain'
[DEBUG] Model output: Connection error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
[DEBUG] Parsed 0 tool call(s)
[Turn 2] AGENT
  Connection error: 404 Client Error: Not Found for url: http://localhost:11434/api/chat
  ────────────────────────────────────────────────────────────────

[Turn 3] USER
  >> 'DOB is May 14th 1990'

In [ ]:
# A6: Zero Balance — no card should be requested
with patch("tool_executor._lookup_account", return_value=ACC1003):

    t = TestRun("A6", "Zero Balance Account (ACC1003) — skip payment")
    t.send("ACC1003")
    t.send("Priya Agarwal")
    t.send("last 4 of my Aadhaar is 2468")

    t.assert_contains("no outstanding", "zero", "₹0", "0.00", "nothing to pay", "all clear", "no balance")
    t.assert_not_contains("card number", "cvv", "expiry", msg="Must NOT ask for card on zero-balance account")
    t.result()

In [ ]:
# A7: ₹ Symbol in Amount
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("A7", "Rupee symbol + comma in amount: ₹1,000.00")
    t.send("ACC1001")
    t.send("Nithin Jain")
    t.send("1990-05-14")
    t.send("₹1,000.00", "amount with rupee symbol and comma")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain")

    t.assert_contains("txn_", "success")
    t.result()

## Group B — Verification Failure

In [ ]:
# B1: Wrong Name
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("B1", "Wrong Name Rejected")
    t.send("ACC1001")
    t.send("my name is John Doe", "completely wrong name")
    t.send("1990-05-14", "correct DOB, but name still wrong")

    t.assert_not_contains("verified", "balance is", "outstanding balance",
                          msg="Must not verify with wrong name")
    t.result()

In [ ]:
# B2: Correct Name, Wrong Secondary Factor
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("B2", "Correct Name, Wrong Secondary Factor")
    t.send("ACC1001")
    t.send("Nithin Jain", "correct name")
    t.send("I was born on 1st January 1999", "wrong DOB")

    t.assert_not_contains("verified", "outstanding balance",
                          msg="Must not verify with wrong DOB")
    t.result()

In [ ]:
# B3: Lock After 3 Failures
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("B3", "Session Lock After 3 Failures")
    t.send("ACC1001")
    t.send("Nithin Jain")
    t.send("2000-01-01", "wrong attempt #1")
    t.send("1999-06-15", "wrong attempt #2")
    t.send("1998-03-20", "wrong attempt #3")
    r_post = t.send("1990-05-14", "real DOB after lock — must be rejected")

    locked = (
        t.agent._session.is_locked
        or any(k in r_post.lower() for k in ("locked", "exceeded", "support", "contact", "maximum"))
    )
    t.assert_true(locked, f"Session should be locked. Last response: {r_post}")
    t.result()

In [ ]:
# B4: Locked Session Stays Locked
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("B4", "Locked Session Rejects All Subsequent Input")
    t.send("ACC1001")
    t.send("Nithin Jain")
    t.send("2000-01-01")
    t.send("1999-06-15")
    t.send("1998-03-20")  # 3rd failure → lock
    r = t.send("okay my real DOB is 1990-05-14", "correct DOB after lock")

    t.assert_true(
        "locked" in r.lower() or "support" in r.lower() or t.agent._session.is_locked,
        f"Expected locked message. Got: {r}"
    )
    t.result()

In [ ]:
# B5: Case-Sensitive Name Matching
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("B5", "Case-Sensitive Name (nithin jain != Nithin Jain)")
    t.send("ACC1001")
    t.send("nithin jain", "all lowercase — must fail strict match")
    t.send("1990-05-14")

    t.assert_not_contains("verified", "outstanding balance",
                          msg="Case-insensitive match must NOT be accepted")
    t.result()

In [ ]:
# B6: Retry Guidance on First Failure
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("B6", "Retry Guidance on First Failure")
    t.send("ACC1001")
    t.send("Nithin Jain")
    r = t.send("1999-01-01", "first wrong attempt")

    has_guidance = any(k in r.lower() for k in
        ("attempt", "try", "remaining", "again", "incorrect", "match", "wrong"))
    t.assert_true(has_guidance, f"Expected retry guidance. Got: {r}")
    t.result()

In [ ]:
# B7: No Sensitive Data on Failure
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("B7", "No Sensitive Data Leaked on Verification Failure")
    t.send("ACC1001")
    t.send("Nithin Jain")
    t.send("1999-01-01", "wrong DOB")

    t.assert_no_sensitive_leak()
    t.result()

In [ ]:
# B8: Pincode as Secondary Factor
with patch("tool_executor._lookup_account", return_value=ACC1002), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("B8", "Pincode as Secondary Factor (ACC1002)")
    t.send("ACC1002")
    t.send("Rajarajeswari Balasubramaniam")
    t.send("my pincode is 400002", "pincode instead of DOB/Aadhaar")
    t.send("540")
    t.send("4532015112830366, 123, 12/2027, Rajarajeswari Balasubramaniam")

    t.assert_contains("txn_", "success")
    t.result()

## Group C — Payment Failure

In [ ]:
# C1: API invalid_card Error
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_INVALID_CARD):
    t = TestRun("C1", "API Returns invalid_card")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("1990-05-14"); t.send("500")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain")

    t.assert_contains("invalid", "card", "re-enter", "again", "incorrect")
    t.assert_not_contains("txn_", "transaction id", msg="Must not show txn ID on failure")
    t.result()

In [ ]:
# C2: API invalid_cvv Error
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_INVALID_CVV):

    t = TestRun("C2", "API Returns invalid_cvv")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("400001"); t.send("500")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain")

    t.assert_contains("cvv", "invalid", "security", "code", "card")
    t.result()

In [ ]:
# C3: API invalid_expiry Error
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_INVALID_EXPIRY):

    t = TestRun("C3", "API Returns invalid_expiry")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("4321"); t.send("500")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain")

    t.assert_contains("expir", "invalid", "card", "date")
    t.result()

In [ ]:
# C4: Account Not Found (404)
with patch("tool_executor._lookup_account",
           side_effect=APIError("Not found", 404, "account_not_found")):

    t = TestRun("C4", "Account Not Found — 404")
    t.send("ACC9999")

    t.assert_contains("not found", "doesn't exist", "no account", "couldn't find", "does not exist")
    t.result()

In [ ]:
# C5: Network Timeout on Lookup
with patch("tool_executor._lookup_account",
           side_effect=APIError("Timeout", 0, "timeout")):

    t = TestRun("C5", "Network Timeout on Account Lookup")
    t.send("ACC1001")

    t.assert_contains("try again", "network", "trouble", "connection", "moment", "timeout")
    t.result()

In [ ]:
# C6: Luhn Failure Blocks API Call
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment") as mock_pay:

    t = TestRun("C6", "Luhn-Invalid Card — process_payment Must Not Be Called")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("1990-05-14"); t.send("500")
    t.send("1234567890123456, 123, 12/2027, Nithin Jain", "fails Luhn checksum")

    t.assert_true(not mock_pay.called, "process_payment was called despite Luhn failure")
    t.assert_contains("invalid", "card number", "incorrect")
    t.result()

In [ ]:
# C7: Expired Card Blocks API Call
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment") as mock_pay:

    t = TestRun("C7", "Expired Card — process_payment Must Not Be Called")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("400001"); t.send("500")
    t.send("4532015112830366, 123, expires 01/2020, Nithin Jain", "expired Jan 2020")

    t.assert_true(not mock_pay.called, "process_payment was called despite expired card")
    t.assert_contains("expired", "expiry", "invalid")
    t.result()

In [ ]:
# C8: Amount Exceeds Balance
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment") as mock_pay:

    t = TestRun("C8", "Amount Exceeds Balance — Blocked Before Payment")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("1990-05-14")
    t.send("I need to pay 99999 rupees", "way over ₹1,250.75 balance")

    t.assert_true(not mock_pay.called, "process_payment was called despite over-limit amount")
    t.assert_contains("exceed", "balance", "too much", "amount", "lower", "1250")
    t.result()

In [ ]:
# C9: Retry After Payment Failure
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", side_effect=[PAY_INVALID_CARD, PAY_SUCCESS]):

    t = TestRun("C9", "Card Error → Retry → Success")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("1990-05-14"); t.send("500")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain", "first attempt — invalid_card")
    r = t.send("let me retry: 4532015112830366, 123, 12/2027, Nithin Jain", "second attempt — success")

    t.assert_true(
        any(k in r.lower() for k in ("txn_", "success", "transaction")),
        f"Expected success on retry. Got: {r}"
    )
    t.result()

In [ ]:
# C10: No Payment Before Verification
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment") as mock_pay:

    t = TestRun("C10", "process_payment Must Not Fire Before Identity Verified")
    t.send("ACC1001")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain", "card before identity verification")

    t.assert_true(not mock_pay.called, "process_payment was called before verification")
    t.result()

## Group D — Edge Cases

In [ ]:
# D1: Leap Year DOB Accepted
with patch("tool_executor._lookup_account", return_value=ACC1004), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("D1", "Leap Year DOB 1988-02-29 Accepted (ACC1004)")
    t.send("ACC1004")
    t.send("Rahul Mehta")
    t.send("I was born on February 29, 1988", "valid leap year date")
    t.send("pay 1000")
    t.send("4532015112830366, 123, 12/2027, Rahul Mehta")

    t.assert_contains("txn_", "success")
    t.result()

In [ ]:
# D2: Off-By-One Leap Year DOB Rejected
with patch("tool_executor._lookup_account", return_value=ACC1004):

    t = TestRun("D2", "Off-By-One Leap Year DOB Rejected (1988-02-28)")
    t.send("ACC1004")
    t.send("Rahul Mehta")
    t.send("1988-02-28", "off by one day — must not match")

    t.assert_not_contains("verified", "balance is", "outstanding balance")
    t.result()

In [ ]:
# D3: Natural Language DOB Formats
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("D3", "Natural Language DOB — 'May 14, 90'")
    t.send("ACC1001"); t.send("Nithin Jain")
    t.send("DOB is May 14, 90", "short year format — doc example")
    t.send("500")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain")

    t.assert_contains("txn_", "success")
    t.result()

In [ ]:
# D4: 'Just clear the full amount' in Natural Language
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("D4", "Full Balance via Natural Language")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("1990-05-14")
    t.send("just clear the full amount", "doc example verbatim")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain")

    t.assert_contains("txn_", "success", "1250")
    t.result()

In [ ]:
# D5: Spaced Card Number
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("D5", "Card Number with Spaces: '4532 0151 1283 0366'")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("400001"); t.send("500")
    t.send("the card number is 4532 0151 1283 0366, CVV is 123, expires 12/2027, Nithin Jain",
           "doc example: card with spaces")

    t.assert_contains("txn_", "success")
    t.result()

In [ ]:
# D6: Two-Digit Expiry Year
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("D6", "Two-Digit Expiry Year: '12/27' → 2027")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("4321"); t.send("500")
    t.send("4532015112830366, 123, expires 12/27, Nithin Jain", "2-digit year doc example")

    t.assert_contains("txn_", "success")
    t.result()

In [ ]:
# D7: Spaced Pincode
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("D7", "Spaced Pincode: '4 0 0 0 0 1'")
    t.send("ACC1001"); t.send("Nithin Jain")
    t.send("pincode? it's 4 0 0 0 0 1", "doc example verbatim")
    t.send("500")
    t.send("4532015112830366, 123, 12/2027, Nithin Jain")

    t.assert_contains("txn_", "success")
    t.result()

In [ ]:
# D8: No Re-ask for Already Given Info
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("D8", "No Re-ask for Info Already Provided")
    t.send("Hi, I'm Nithin Jain and my account is ACC1001", "name + account in one message")
    r = t.send("4321", "secondary factor — agent must not re-ask name")

    name_reasked = "full name" in r.lower() or "what is your name" in r.lower() or "your name" in r.lower()
    t.assert_false(name_reasked, f"Agent re-asked for name. Got: {r}")
    t.result()

In [ ]:
# D9: Empty / Whitespace Input
with patch("tool_executor._lookup_account", return_value=ACC1001):

    t = TestRun("D9", "Whitespace-Only Input Handled Gracefully")
    t.send("ACC1001")
    r = t.send("   ", "whitespace-only input")

    t.assert_true(isinstance(r, str) and len(r.strip()) > 0, "Response must not be empty")
    t.result()

In [ ]:
# D10: CVV Spoken as Words
with patch("tool_executor._lookup_account", return_value=ACC1001), \
     patch("tool_executor._process_payment", return_value=PAY_SUCCESS):

    t = TestRun("D10", "CVV Spoken as Words: 'one two three'")
    t.send("ACC1001"); t.send("Nithin Jain"); t.send("1990-05-14"); t.send("500")
    t.send("CVV is one two three, card 4532015112830366, expires 12/2027, Nithin Jain",
           "doc example: CVV as spoken words")

    t.assert_contains("txn_", "success")
    t.result()